- 基于时序差分（Temporal-Difference, TD）的方法：利用值函数（Value Function）来估计优势，考虑每个时间步（token）的奖励，例如 `GAE`。
- 基于蒙特卡洛（Monte Carlo）的方法：使用完整的序列奖励（outcome reward），通常会结合各种基线（baseline）来减小方差，例如 GRPO, RLOO 等大部分算法。这类算法的优势函数在单个序列的所有 token 上通常是常数。
- Token 级别优势估计：这类方法为序列中的每一步（Token）计算优势 $A_t$，通常依赖于价值函数 $V_t$ 或者蒙特卡洛回报 $G_t$
    - GAE
    - REINFORCE++ (RF++)
    - REMAX
- 序列级别（结果导向）优势估计 (Outcome-Based)
    - GRPO
    - REINFORCE++-Baseline
    - RLOO (Reinforcement Learning with Leave-One-Out)

### GAE

### REINFORCE++, REINFORCE++-bl

- RF++
    - 不依赖分组，它直接使用蒙特卡洛回报（discounted future returns）作为优势，并对其进行 whiten 处理。
    - 时间步 $t$，回报 $R_t=\sum_{k=t}^{T-1}\gamma^{k-t}r_k$
    - $\hat A_t=\text{whiten}(R_t)$
- RF++-bl
    - 该算法与 GRPO 非常相似，也使用组内平均奖励作为基线。主要区别在于计算出 (序列奖励 - 组均值) 后，会进行 whiten 操作

In [1]:
import numpy as np

In [3]:
token_level_rewards = np.zeros((5, 8))

In [6]:
token_level_rewards[:, -1] = np.random.randint(0, 2, 5)

In [7]:
token_level_rewards

array([[0., 0., 0., 0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 0., 0., 0., 1.]])

In [14]:
gamma = 0.9
running_return = 0
returns = np.zeros_like(token_level_rewards)
response_mask = np.ones_like(token_level_rewards)
for t in reversed(range(token_level_rewards.shape[1])):
    running_return = token_level_rewards[:, t] + gamma * running_return
    returns[:, t] = running_return
    # Reset after EOS
    running_return = running_return * response_mask[:, t]
returns

array([[0.4782969, 0.531441 , 0.59049  , 0.6561   , 0.729    , 0.81     ,
        0.9      , 1.       ],
       [0.4782969, 0.531441 , 0.59049  , 0.6561   , 0.729    , 0.81     ,
        0.9      , 1.       ],
       [0.       , 0.       , 0.       , 0.       , 0.       , 0.       ,
        0.       , 0.       ],
       [0.4782969, 0.531441 , 0.59049  , 0.6561   , 0.729    , 0.81     ,
        0.9      , 1.       ],
       [0.4782969, 0.531441 , 0.59049  , 0.6561   , 0.729    , 0.81     ,
        0.9      , 1.       ]])

### GRPO vs. RLOO